In [ ]:
# Fix the pandas version first to keep the system happy
!pip install -q "pandas<3.0"

# Install Whisper dependencies without touching your working Torch install
!pip install -q transformers datasets evaluate jiwer librosa accelerate

# Explicitly uninstall existing torch, torchaudio, torchvision to clear any conflicts
# !pip uninstall -y torch torchaudio torchvision

# Install torch, torchaudio, and torchvision for CUDA 12.1 (compatible with 12.8)
# This aligns with the nvcc version observed on the system.
!pip install torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu121

# Verify nvcc version
!nvcc --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 48.4 MB/s eta 0:00:00
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 51.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 42.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/82

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version torch is using: {torch.version.cuda}")

import torchaudio
import torch

print(f"✅ Torchaudio version: {torchaudio.__version__}")
# Removed the problematic line: print(f"✅ Backend: {torchaudio.list_audio_backends()}")

# Simple test to ensure the library can actually compute
try:
    dummy_input = torch.randn(1, 16000).cuda()
    spec = torchaudio.transforms.Spectrogram().cuda()(dummy_input)
    print("🚀 Hardware acceleration test: SUCCESS")
except Exception as e:
    print(f"❌ Hardware acceleration test: FAILED - {e}")

PyTorch Version: 2.5.1+cu121
CUDA available: True
Current device: Tesla T4
CUDA version torch is using: 12.1
✅ Torchaudio version: 2.5.1+cu121
🚀 Hardware acceleration test: SUCCESS


In [ ]:
!pip install -q faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 22.6 MB/s eta 0:00:00


In [ ]:
import os
import re
import gc
import json
import math
import random
import string
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import librosa
# import torchaudio
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from jiwer import wer
from faster_whisper import WhisperModel
from transformers import AutoModel
from huggingface_hub import login

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RUNNING_IN_KAGGLE = True  # Set True only after moving notebook into Kaggle

In [ ]:
# -------------------------------
# Paths and runtime config
# -------------------------------
import os
from huggingface_hub import login

# --- AUTHENTICATION ---
HF_TOKEN = None

if RUNNING_IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("💡 Kaggle Secret 'HF_TOKEN' not found.")
else:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        print("💡 Colab Secret 'HF_TOKEN' not found.")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ Authenticated with Hugging Face Hub")
else:
    print("⚠️ No token found. Using public access.")


if RUNNING_IN_KAGGLE:
    INPUT_PATH = "/kaggle/input/competitions/multilingual-speech-recognition"
    #/kaggle/input/competitions/multilingual-speech-recognition
    WORK_DIR = "/kaggle/working/nppe2_hybrid"
    TRAIN_CSV = os.path.join(INPUT_PATH, "train.csv")
    TEST_CSV = os.path.join(INPUT_PATH, "test.csv")
    TRAIN_DIR = os.path.join(INPUT_PATH, "competition_data", "train")
    TEST_DIR = os.path.join(INPUT_PATH, "competition_data", "test")
    SUBMISSION_PATH = "/kaggle/working/submission.csv"
else:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    ZIP_PATH = "/content/drive/MyDrive/multilingual-speech-recognition.zip"  # change if needed
    EXTRACT_ROOT = "/content/competition_data"
    WORK_DIR = "/content/nppe2_hybrid"

    if not os.path.exists(EXTRACT_ROOT):
        !unzip -q "$ZIP_PATH" -d "$EXTRACT_ROOT"
        print("Dataset extracted to", EXTRACT_ROOT)

    TRAIN_CSV = os.path.join(EXTRACT_ROOT, "train.csv")
    TEST_CSV = os.path.join(EXTRACT_ROOT, "test.csv")
    TRAIN_DIR = os.path.join(EXTRACT_ROOT, "competition_data", "train")
    TEST_DIR = os.path.join(EXTRACT_ROOT, "competition_data", "test")
    SUBMISSION_PATH = "/content/submission.csv"

os.makedirs(WORK_DIR, exist_ok=True)
print("TRAIN_CSV:", TRAIN_CSV)
print("TEST_CSV:", TEST_CSV)
print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR:", TEST_DIR)
print("WORK_DIR:", WORK_DIR)

💡 Colab Secret 'HF_TOKEN' not found.
⚠️ No token found. Using public access.
Mounted at /content/drive
Dataset extracted to /content/competition_data
TRAIN_CSV: /content/competition_data/train.csv
TEST_CSV: /content/competition_data/test.csv
TRAIN_DIR: /content/competition_data/competition_data/train
TEST_DIR: /content/competition_data/competition_data/test
WORK_DIR: /content/nppe2_hybrid


In [ ]:
# -------------------------------
# Main knobs (pipeline controls)
# -------------------------------

VAL_SIZE = 0.10  
# used in train/validation split → controls how much data is used for local WER evaluation

EVAL_MAX_ROWS = 220  
# used during validation loop → limits number of samples for faster WER computation

USE_CACHE = True  
# used in LID + transcription steps → decides whether to load saved results instead of recomputing

RUN_VALIDATION = True  
# used in main pipeline flow → decides whether to run validation step before test inference


# -------------------------------
# Model selection
# -------------------------------

LID_MODEL_ID = "small"  
# used in language detection function → lightweight Whisper model for predicting language

WHISPER_MODEL_ID = "large-v3-turbo"  
# used in main ASR transcription → high-accuracy Whisper model for EN/HI/TA languages

INDIC_MODEL_ID = "ai4bharat/indic-conformer-600m-multilingual"  
# used in routing fallback → primary model for Indic languages not handled by Whisper

INDIC_FALLBACK_ID = "sunilmahendrakar/indic-conformer-600m-multilingual"  
# used if INDIC_MODEL_ID fails → backup model for robustness


# -------------------------------
# Whisper decoding settings
# -------------------------------

WHISPER_BEAM_SIZE = 5  
# used inside whisper.transcribe() → controls beam search width for better decoding

WHISPER_COMPUTE_TYPE = "float16" if torch.cuda.is_available() else "int8"  
# used when loading Whisper → controls precision (fp16 on GPU, int8 on CPU for efficiency)

WHISPER_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  
# used when initializing Whisper → decides whether model runs on GPU or CPU


# -------------------------------
# Cache file paths
# -------------------------------

LID_CACHE_VAL = os.path.join(WORK_DIR, "val_lid.csv")  
# used in validation LID step → stores detected languages to avoid recomputation

LID_CACHE_TEST = os.path.join(WORK_DIR, "test_lid.csv")  
# used in test LID step → saves language predictions for test data

VAL_PRED_CACHE = os.path.join(WORK_DIR, "val_preds.csv")  
# used after validation inference → stores predicted transcripts for WER reuse

TEST_PRED_CACHE = os.path.join(WORK_DIR, "test_preds.csv")  
# used after test inference → stores final predictions for submission reuse

In [ ]:
# -------------------------------
# Helpers
# -------------------------------

def resolve_audio_path(audio_name: str, split: str) -> str:
    # Given an audio filename + split (train/test), find the correct file path from multiple possible directory structures
    base_dir = TRAIN_DIR if split == "train" else TEST_DIR
    
    # Try different possible locations where the audio file might exist (handles messy Kaggle paths)
    candidates = [
        os.path.join(base_dir, audio_name),                         # standard location
        os.path.join(Path(base_dir).parent, audio_name),            # parent directory
        os.path.join(Path(base_dir).parent, split, audio_name),     # parent/split/audio
        os.path.join(Path(TRAIN_CSV).parent, "competition_data", split, audio_name),  # Kaggle competition_data format
        os.path.join(Path(TRAIN_CSV).parent, split, audio_name),    # alternative split folder
    ]
    
    for path in candidates:
        if os.path.exists(path):
            return path
    
    raise FileNotFoundError(f"Could not resolve path for {audio_name} ({split})")


def normalize_reference(text: str) -> str:
    # Clean ground-truth transcripts → remove extra spaces and normalize formatting
    return re.sub(r"\s+", " ", str(text).strip())


def clean_submission_text(text: str, route_group: str) -> str:
    # Clean predicted ASR output before evaluation/submission
    
    text = normalize_reference(text)  # remove extra spaces
    
    if route_group == "en":
        text = text.lower()  # normalize case for English (WER sensitive to case)
    
    text = re.sub(r"[_]", " ", text)  # replace underscores with space
    
    # Remove unwanted characters but keep:
    # - word characters
    # - whitespace
    # - Indian script ranges (Devanagari, Tamil, etc.)
    text = re.sub(r"[^\w\sऀ-ॿ஀-௿]", " ", text, flags=re.UNICODE)
    
    text = re.sub(r"\s+", " ", text).strip()  # final cleanup
    
    return text


def detect_script(text: str) -> str:
    # Detect which writing script the text belongs to (Latin, Devanagari, Tamil, etc.)
    
    counts = {}
    
    for ch in str(text):
        if not ch.isalpha():  # ignore non-letter characters
            continue
        
        name = unicodedata.name(ch, "")  # get Unicode name of character
        
        # Check which script the character belongs to
        for script in ["LATIN", "DEVANAGARI", "TAMIL", "ARABIC", "BENGALI", "TELUGU", "MALAYALAM"]:
            if script in name:
                counts[script] = counts.get(script, 0) + 1
                break
    
    # Return script with highest frequency
    return max(counts, key=counts.get) if counts else "OTHER"


def route_language(pred_lang: str) -> str:
    # Map predicted language into routing groups for ASR models
    
    pred_lang = (pred_lang or "").lower()
    
    if pred_lang == "en":
        return "en"   # route to English Whisper
    
    if pred_lang in {"hi", "ur", "mr", "ne", "mai", "doi", "sa"}:
        return "hi"   # group Indo-Aryan languages with Hindi model
    
    if pred_lang in {"ta", "ml"}:
        return "ta"   # group Dravidian languages with Tamil model
    
    return "fallback"  # all others → Indic fallback model


def clear_memory(*objs):
    # Free memory after heavy model usage (important for GPU)
    
    for obj in objs:
        del obj  # delete object references
    
    gc.collect()  # force Python garbage collection
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # clear GPU memory cache

In [ ]:
# -------------------------------
# Load metadata and build split
# -------------------------------
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

# Auto-detect column names so the notebook remains robust to small schema changes.
audio_col = "audio" if "audio" in train_df.columns else train_df.columns[0]
text_col = "text" if "text" in train_df.columns else train_df.columns[-1]

train_df[text_col] = train_df[text_col].map(normalize_reference)
train_df["audio_path"] = train_df[audio_col].map(lambda x: resolve_audio_path(x, "train"))
test_df["audio_path"] = test_df[audio_col].map(lambda x: resolve_audio_path(x, "test"))

train_df["script"] = train_df[text_col].map(detect_script)
script_counts = train_df["script"].value_counts()
train_df["script_bucket"] = train_df["script"].where(train_df["script"].map(script_counts) >= 5, "RARE")

tr_df, va_df = train_test_split(
    train_df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_df["script_bucket"],
)
tr_df = tr_df.reset_index(drop=True)
va_df = va_df.reset_index(drop=True)

# Reduce validation size while preserving language/script diversity:
# eg: Total rows = 1000
# EVAL_MAX_ROWS = 200
# Group sizes:
# Latin → 500 → gets 100 samples
# Devanagari → 300 → gets 60
# Tamil → 200 → gets 40
if EVAL_MAX_ROWS and len(va_df) > EVAL_MAX_ROWS:
    va_df = va_df.groupby("script_bucket", group_keys=False).apply(
        lambda x: x.sample(max(1, round(EVAL_MAX_ROWS * len(x) / len(va_df))), random_state=SEED)
    ).reset_index(drop=True)
    va_df = va_df.head(EVAL_MAX_ROWS).copy()

print("train rows:", len(train_df))
print("validation rows used:", len(va_df))
print(train_df["script_bucket"].value_counts())
# train rows: 2000
# validation rows used: 200
# script_bucket
# LATIN         998
# DEVANAGARI    508
# TAMIL         494

train rows: 2000
validation rows used: 200
script_bucket
LATIN         998
DEVANAGARI    508
TAMIL         494
Name: count, dtype: int64


In [ ]:
# -------------------------------
# Step 1: language detection with a small Whisper model
# -------------------------------
def add_lid_predictions(df: pd.DataFrame, cache_path: str) -> pd.DataFrame:
    # adds 2 new columns:
    # 1) predicted language
    # 2) predicted language confidence

    if USE_CACHE and os.path.exists(cache_path):
        cached = pd.read_csv(cache_path)

        # Check cache validity:
        # same number of rows and required columns present
        if len(cached) == len(df) and set([audio_col, "pred_lang", "pred_lang_prob"]).issubset(cached.columns):
            # Merge cached language predictions back into current dataframe using audio filename/path
            merged = df.merge(cached[[audio_col, "pred_lang", "pred_lang_prob"]], on=audio_col, how="left")

            # If every row got a valid predicted language, return cached result directly
            if merged["pred_lang"].notna().all():
                print(f"Loaded cached LID from {cache_path}")
                return merged

    # Small Whisper is used only for language detection to keep this stage fast.
    lid_model = WhisperModel(LID_MODEL_ID, device=WHISPER_DEVICE, compute_type=WHISPER_COMPUTE_TYPE)

    pred_langs, pred_probs = [], []
    # Lists to store predicted language codes and confidence scores

    for path in tqdm(df["audio_path"].tolist(), desc=f"LID -> {Path(cache_path).name}"):
        # Loop through every audio file path with a progress bar

        segments, info = lid_model.transcribe(
            path,
            task="transcribe",
            beam_size=1,
            best_of=1,
            temperature=0.0,
            condition_on_previous_text=False,
            vad_filter=False,
        ) #segment 1: start=0.0, end=1.2, text="Hello everyone."
        # Run Whisper on the audio file
        # We do not care about full transcript here, only the metadata `info`
        # which contains language and language probability

        _ = list(segments)
        # Force the generator to fully run so info gets populated correctly

        pred_langs.append(getattr(info, "language", "unknown"))
        # Save predicted language code, fallback = "unknown"

        pred_probs.append(float(getattr(info, "language_probability", 0.0)))
        # Save confidence/probability of the predicted language, fallback = 0.0

    out = df.copy()
    # Create a copy of the dataframe so original is not modified directly

    out["pred_lang"] = pred_langs
    # Add predicted language column

    out["pred_lang_prob"] = pred_probs
    # Add predicted language confidence column

    out[[audio_col, "pred_lang", "pred_lang_prob"]].to_csv(cache_path, index=False)
    # Save these LID results to CSV so future runs can reuse them

    clear_memory(lid_model)
    # Free GPU/CPU memory used by the Whisper LID model

    return out
    # Return dataframe with language predictions added


if RUN_VALIDATION:
    va_df = add_lid_predictions(va_df, LID_CACHE_VAL)
    # Add language predictions to validation set if validation is enabled

test_df = add_lid_predictions(test_df, LID_CACHE_TEST)
# Always add language predictions to test set, because test inference is needed for submission

LID -> val_lid.csv:   0%|          | 0/200 [00:00<?, ?it/s]

LID -> test_lid.csv:   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
# -------------------------------
# Step 2: model wrappers
# -------------------------------

def transcribe_with_whisper(paths, forced_language=None, desc="Whisper"):
    model = WhisperModel(WHISPER_MODEL_ID, device=WHISPER_DEVICE, compute_type=WHISPER_COMPUTE_TYPE)
    # Load Whisper model (used for actual ASR)

    outputs = []
    for path in tqdm(paths, desc=desc):
        segments, info = model.transcribe(
            path,
            language=forced_language,            # force language (from LID) to improve accuracy
            task="transcribe",                   # perform transcription (not translation)
            beam_size=WHISPER_BEAM_SIZE,         # beam search for better decoding
            best_of=WHISPER_BEAM_SIZE,           # number of candidate outputs
            temperature=0.0,                     # deterministic output
            condition_on_previous_text=False,    # avoid context carryover
            vad_filter=True,                     # remove silence segments
            vad_parameters={"min_silence_duration_ms": 500},  # silence threshold
        )

        text = " ".join(seg.text.strip() for seg in segments).strip()
        # Combine segment-wise outputs into one sentence: ["Hello", "world", "how are you"] → "Hello world how are you"

        outputs.append(text)
        # Store transcript

    clear_memory(model)
    # Free GPU/CPU memory

    return outputs
    # Return list of transcripts


def load_indic_model():
    # Load Indic ASR model with fallback support

    model_id_to_try = [INDIC_MODEL_ID]
    # Start with primary model

    if INDIC_FALLBACK_ID:
        model_id_to_try.append(INDIC_FALLBACK_ID)
        # Add backup model if available

    last_error = None
    for model_id in model_id_to_try:
        try:
            print("Loading Indic model:", model_id)

            kwargs = {"trust_remote_code": True}
            # Allow custom HF model code
            if HF_TOKEN:
                kwargs["token"] = HF_TOKEN
                
            model = AutoModel.from_pretrained(model_id, **kwargs)

            return model, model_id
            # Return first successfully loaded model

        except Exception as exc:
            print(f"Failed to load {model_id}: {exc}")
            last_error = exc
            # Try next model if failed

    raise RuntimeError(f"Could not load any Indic model. Last error: {last_error}")
    # Fail if no model loads


def load_wav_16k_mono(path: str):
    # Load audio and standardize format

    wav, sr = torchaudio.load(path)
    # Load waveform + sample rate

    wav = torch.mean(wav, dim=0, keepdim=True)
    # Convert to mono (combine channels)

    if sr != 16000:
        wav = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(wav)
        # Resample to 16kHz (model requirement)

    return wav
    # Return processed waveform


def transcribe_with_indic(paths, lang_code, desc="Indic"):
    # Transcribe audio using Indic ASR model

    model, loaded_id = load_indic_model()
    # Load primary or fallback model

    outputs = []
    for path in tqdm(paths, desc=f"{desc} [{lang_code}]"):
        wav = load_wav_16k_mono(path)
        # Convert audio to 16kHz mono

        text = model(wav, lang_code, "rnnt")
        # Run ASR with language code + RNNT  (Recurrent Neural Network Transducer) decoding
        # Whisper uses a sequence-to-sequence architecture, and the Indic model uses RNNT. CTC is not used in this pipeline.
        # While CTC assumes that outputs at each time frame are independent, RNN-T uses the predictor network to model dependencies between output tokens
        outputs.append(text if isinstance(text, str) else str(text))
        # Store output as string

    clear_memory(model)
    print("Indic model used:", loaded_id)

    return outputs # transcripts

In [ ]:
# -------------------------------
# Step 3: routed inference
# -------------------------------
def run_routed_inference(df: pd.DataFrame, cache_path: str = None) -> pd.DataFrame:
    # Run ASR by routing each audio to the chosen model path

    if cache_path and USE_CACHE and os.path.exists(cache_path):
        cached = pd.read_csv(cache_path)

        required = {audio_col, "pred_text", "route_group", "pred_lang", "pred_lang_prob"}
        # Minimum columns needed for a valid cached result

        if len(cached) == len(df) and required.issubset(cached.columns):
            # Use cache only if row count and required columns match
            print(f"Loaded cached predictions from {cache_path}")
            return cached

    work = df.copy()
    work["route_group"] = work["pred_lang"].map(route_language)
    # Convert predicted language into routing bucket: en / hi / ta / fallback

    work["pred_text"] = ""
    # Empty column to store final transcript predictions

    english_mask = work["route_group"] == "en"
    # Select rows routed to English path
    hindi_mask = work["route_group"] == "hi"
    tamil_mask = work["route_group"] == "ta"
    fallback_mask = work["route_group"] == "fallback"

    # Apply each model only to its assigned subset for both speed and quality.
    if english_mask.any():
        # Run only English-routed samples through Whisper English
        texts = transcribe_with_whisper(
            work.loc[english_mask, "audio_path"].tolist(),
            forced_language="en",          # force English decoding
            desc="Whisper English",        # progress bar label
        )
        work.loc[english_mask, "pred_text"] = texts
        # Store English predictions back into dataframe

    if hindi_mask.any():
        texts = transcribe_with_indic(
            work.loc[hindi_mask, "audio_path"].tolist(),
            lang_code="hi",                # decode as Hindi route
            desc="Indic Hindi route",      # progress bar label
        )
        work.loc[hindi_mask, "pred_text"] = texts

    if tamil_mask.any():
        texts = transcribe_with_indic(
            work.loc[tamil_mask, "audio_path"].tolist(),
            lang_code="ta",                # decode as Tamil route
            desc="Indic Tamil route",      
        )
        work.loc[tamil_mask, "pred_text"] = texts

    if fallback_mask.any():
        texts = transcribe_with_whisper(
            work.loc[fallback_mask, "audio_path"].tolist(),
            forced_language=None,          # let Whisper infer language itself
            desc="Whisper fallback",       
        )
        work.loc[fallback_mask, "pred_text"] = texts

    # Final normalization keeps output format consistent before scoring/submission.
    work["pred_text"] = [clean_submission_text(t, r) for t, r in zip(work["pred_text"], work["route_group"])]
    # Clean each prediction using route-aware text normalization

    if cache_path:
        # Save useful columns so future runs can skip recomputation
        keep_cols = [c for c in [audio_col, "audio_path", text_col, "script_bucket", "pred_lang", "pred_lang_prob", "route_group", "pred_text"] if c in work.columns]
        work[keep_cols].to_csv(cache_path, index=False)

    return work
    # Return dataframe with route group + final cleaned predictions
# | audio        | audio_path         | pred_lang | pred_lang_prob | route_group | pred_text                     |
# |--------------|--------------------|-----------|----------------|-------------|-------------------------------|
# | a1.wav       | /data/a1.wav       | en        | 0.97           | en          | this is a sample sentence     |
# | a2.wav       | /data/a2.wav       | hi        | 0.92           | hi          | यह एक उदाहरण वाक्य है         |

In [ ]:
# # --- Updated Token and Timeout Config ---
# import os
# from huggingface_hub import login
# from google.colab import userdata

# # 2. Force a longer timeout for large model downloads
# os.environ["HUGGINGFACE_HUB_READ_TIMEOUT"] = "300" # 5 minutes
# os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"    # Speeds up downloads significantly
# import torch
# import torchaudio  # 🚨 Added this to fix your error
# from tqdm.auto import tqdm
# from transformers import AutoModel, AutoProcessor
# from faster_whisper import WhisperModel

# # -------------------------------
# # Step 2: Model wrappers
# # -------------------------------

# def load_wav_16k_mono(path: str):
#     # Load audio and convert it to 16 kHz mono waveform.
#     # Ensure torchaudio is available in this scope
#     wav, sr = torchaudio.load(path)
#     wav = torch.mean(wav, dim=0, keepdim=True)
#     if sr != 16000:
#         wav = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(wav)
#     return wav

# def transcribe_with_whisper(paths, forced_language=None, desc="Whisper"):
#     # Transcribe a batch of audio files with Whisper and return text outputs.
#     model = WhisperModel(WHISPER_MODEL_ID, device=WHISPER_DEVICE, compute_type=WHISPER_COMPUTE_TYPE)
#     outputs = []
#     for path in tqdm(paths, desc=desc):
#         segments, info = model.transcribe(
#             path,
#             language=forced_language,
#             task="transcribe",
#             beam_size=WHISPER_BEAM_SIZE,
#             temperature=0.0,
#             vad_filter=True,
#         )
#         text = " ".join(seg.text.strip() for seg in segments).strip()
#         outputs.append(text)
#     clear_memory(model)
#     return outputs

# def transcribe_with_indic(paths, lang_code, desc="Indic"):
#     # Transcribe with Indic model and keep row order even if a file fails.
#     model, loaded_id = load_indic_model()

#     # Emergency Plan B: Use Whisper if Indic model fails
#     if loaded_id == "whisper_fallback":
#         return transcribe_with_whisper(paths, forced_language=lang_code, desc=f"Whisper-Backup [{lang_code}]")

#     outputs = []
#     for path in tqdm(paths, desc=f"{desc} [{lang_code}]"):
#         try:
#             wav = load_wav_16k_mono(path)
#             # Use RNNT for better WER in this competition
#             text = model(wav, lang_code, "rnnt")
#             outputs.append(text if isinstance(text, str) else str(text))
#         except Exception as e:
#             print(f"Error processing {path}: {e}")
#             outputs.append("") # Keep the row so the CSV alignment doesn't break

#     clear_memory(model)
#     return outputs

In [ ]:
!pip install torchaudio


In [ ]:
import torchaudio
# Import audio library (needed if validation path loads audio/model wrappers)

# -------------------------------
# Optional validation
# -------------------------------
if RUN_VALIDATION:
    print("🚀 Starting Routed Validation...")
    val_pred_df = run_routed_inference(va_df, VAL_PRED_CACHE)
    # Run full routed ASR pipeline on validation set and optionally cache predictions

    val_pred_df[text_col] = val_pred_df[text_col].map(normalize_reference)
    # Clean/normalize ground-truth transcripts before WER calculation

    overall_wer = wer(val_pred_df[text_col].tolist(), val_pred_df["pred_text"].tolist())
    # Compute overall Word Error Rate between reference text and predicted text

    print("\nOverall validation WER:", round(overall_wer, 4)) # WER: 0.6232
    print("\nRoute counts:")
    # Print how many samples went through each routing branch
    print(val_pred_df["route_group"].value_counts(dropna=False))
    # Show counts for en / hi / ta / fallback routes : route_group # en 99 # hi 52 # ta 45 # fallback 4

    display(val_pred_df[[audio_col, text_col, "pred_lang", "route_group", "pred_text"]].head(10))
    # Display first 10 validation rows with reference, predicted language, route, and final prediction




🚀 Starting Routed Validation...


Whisper English:   0%|          | 0/99 [00:00<?, ?it/s]

Loading Indic model: ai4bharat/indic-conformer-600m-multilingual


config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

model_onnx.py: 0.00B [00:00, ?B/s]

...

Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Indic Hindi route [hi]:   0%|          | 0/52 [00:00<?, ?it/s]

In [ ]:
# -------------------------------
# Final test submission
# -------------------------------
test_pred_df = run_routed_inference(test_df, TEST_PRED_CACHE)
submission = test_pred_df[[audio_col]].copy()
submission["text"] = test_pred_df["pred_text"]
submission.to_csv(SUBMISSION_PATH, index=False)

print("Saved submission to:", SUBMISSION_PATH)
print(submission.head(10))
print("Route counts on test:")
print(test_pred_df["route_group"].value_counts(dropna=False))
print("Predicted language counts on test:")
print(test_pred_df["pred_lang"].value_counts(dropna=False).head(20))